# Optimized mode pipeline — pilot only

Run a **256-window pilot**, estimate the full comparison's runtime, and stop. This notebook does not launch a full run or population scatter plots.

The optimized utilities preserve all candidate lags, thresholds, tie-breakers, central-mode gates, and quantile-line metrics. The empty-lag fix is included. `reference_utils.py` is the frozen corrected baseline.

Keep this folder together. Dependencies: NumPy, pandas, SciPy, Matplotlib, IPython; tqdm and threadpoolctl are optional.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

## Settings

Use the TSCC input paths below. The smaller available easy-agreement comparison is selected by window count.

`N_WORKERS=8` requests eight independent processes, capped by CPU affinity or the Slurm allocation. Set it to `1` to measure algorithmic speed independently. Numerical-library threads are limited to one per worker.

The pilot samples windows evenly through the comparison rather than taking only the first genomic region. Its exact indices are saved. An optional eight-window baseline check precedes the pilot.

In [2]:
PIPELINE_DIR = Path.cwd().resolve()
if not (PIPELINE_DIR / "pipeline_runner.py").is_file():
    PIPELINE_DIR = PIPELINE_DIR / "optimized_pipeline"

INPUT_DIR = Path("/tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/processed_inputs")
COMPARISON_TABLE = Path("/tscc/nfs/home/kflanagan/projects/plip_plop/matrix_correspondance/select_comparisons/encode_initial_30_comparisons.tsv")
UTILS_PATH = PIPELINE_DIR / "utils.py"
REFERENCE_UTILS_PATH = PIPELINE_DIR / "reference_utils.py"
OUTPUT_ROOT = PIPELINE_DIR / "pilot_results"

EASY_AGREEMENT_PAIRS = {("PUM1", "PUM2"), ("FUS", "TAF15")}
COMPARISON_FILE = None

SMOOTHING_SIGMA = 2
ABUNDANCE_P = 2

ISLAND_PARAMETERS = dict(
    seed_threshold_fraction=0.4, growth_threshold_fraction=0.2,
    max_weak_distance=5, gap_tolerance=2, shape_balance_power=1.05,
    ridge_fraction=0.6,
    split_prominence_fraction=0.15, split_min_peak_fraction=0.5,
    split_min_peak_distance=8, split_min_seed_area=5,
    displacement_fold_threshold=2.0, mode_score_fraction=0.33
)

QUANTILE_PARAMETERS = dict(n_points=128, signal_influence=1.0, signal_power=1.0, smoothing_sigma=0.0)
ANGLE_QUANTILES = (0.20, 0.80)

PILOT_WINDOWS = 256
BASELINE_CHECK_WINDOWS = 8
N_WORKERS = 8
CHECKPOINT_WINDOWS = 100
CHECKPOINT_SECONDS = 30
STOP_ON_ERROR = True

DBSCAN_EPS = 0.65
DBSCAN_MIN_SAMPLES = 10

In [3]:
if not (PIPELINE_DIR / "pipeline_runner.py").is_file():
    raise FileNotFoundError("Set PIPELINE_DIR to the unpacked optimized_pipeline folder.")
sys.path.insert(0, str(PIPELINE_DIR))
import pipeline_runner

if Path(pipeline_runner.__file__).resolve() != (PIPELINE_DIR / "pipeline_runner.py").resolve():
    raise RuntimeError("A different pipeline_runner is already imported. Restart this kernel.")
analysis_utils = pipeline_runner.load_utils(UTILS_PATH)
if not hasattr(analysis_utils, "_LagScoringContext"):
    raise RuntimeError("The selected utils.py is not the optimized copy.")

configuration = dict(abundance_p=ABUNDANCE_P, smoothing_sigma=SMOOTHING_SIGMA,
    island_parameters=ISLAND_PARAMETERS, quantile_parameters=QUANTILE_PARAMETERS,
    angle_quantiles=ANGLE_QUANTILES, checkpoint_windows=CHECKPOINT_WINDOWS,
    checkpoint_seconds=CHECKPOINT_SECONDS, stop_on_error=STOP_ON_ERROR)
print(f"Optimized utilities: {UTILS_PATH}")

Optimized utilities: /tscc/projects/ps-yeolab3/kflanagan/plip_plop/matrix_speed_up_test/optimized_pipeline/utils.py


## Select and load one comparison

Candidate sizes are read from array headers. The selected archive is decompressed once. Enrichment uses all 300 positions. Old 600-position archives are rejected.

In [4]:
def npz_array_shape(path, key):
    """Read an array header without decompressing its signal values."""
    with zipfile.ZipFile(path) as archive, archive.open(f"{key}.npy") as handle:
        version = np.lib.format.read_magic(handle)
        if version == (1, 0):
            shape, _, _ = np.lib.format.read_array_header_1_0(handle)
        elif version == (2, 0):
            shape, _, _ = np.lib.format.read_array_header_2_0(handle)
        else:
            raise ValueError(f"Unsupported NPY header version {version}: {path}")
    return shape


def comparison_candidates(table_path, input_dir):
    table = pd.read_csv(table_path, sep="\t")
    rows = []
    for comparison_id, row in table.iterrows():
        pair = (row["experiment_A"].split("_")[0], row["experiment_B"].split("_")[0])
        if pair not in EASY_AGREEMENT_PAIRS:
            continue
        path = input_dir / f'{row["experiment_A"]}_{row["experiment_B"]}.npz'
        candidate = dict(comparison_id=int(comparison_id), category=row["category"],
                         experiment_A=row["experiment_A"], experiment_B=row["experiment_B"],
                         path=str(path), n_windows=None, positions=None, status="missing")
        if path.is_file():
            try:
                shape = npz_array_shape(path, "ip_signals_A")
                if len(shape) != 3:
                    raise ValueError(f"Expected a 3D signal array; found {shape}")
                candidate.update(n_windows=shape[0], positions=shape[2], size_mb=path.stat().st_size / 1e6,
                                 status="ready" if shape[2] == 300 and shape[0] > 0 else "wrong_width_or_empty")
            except (ValueError, KeyError, OSError, zipfile.BadZipFile) as exc:
                candidate.update(status="unreadable", error=str(exc))
        rows.append(candidate)
    return pd.DataFrame(rows)

In [5]:
phase_timings = {}
started = perf_counter()
candidates = comparison_candidates(COMPARISON_TABLE, INPUT_DIR)
phase_timings["candidate_scan_seconds"] = perf_counter() - started
display(candidates)

if candidates.empty:
    raise ValueError("No easy-agreement pair was found in the comparison table.")

available = candidates[candidates["status"] == "ready"].copy()
if COMPARISON_FILE is not None:
    available = available[available["path"].map(lambda path: Path(path).name == Path(COMPARISON_FILE).name)]
if available.empty:
    raise ValueError("No eligible 300-nt archive is available. Check the candidate table and regenerate any old 600-nt inputs.")

selected = available.sort_values(["n_windows", "path"]).iloc[0]
INPUT_PATH = Path(selected["path"])
comparison_key = f'{selected["experiment_A"]}__{selected["experiment_B"]}'

print(f"Selected: {comparison_key}")
print(f'Windows: {int(selected["n_windows"]):,}; positions per window: 300')
print("Selection uses window count, not compressed file size.")

,comparison_id,category,experiment_A,experiment_B,path,n_windows,positions,status,size_mb
0,0,similar,FUS_HepG2_ENCSR464OSH,TAF15_HepG2_ENCSR841EQA,/tscc/lustre/ddn/scratch/kflanagan/plip_plop_r...,17981,300,ready,7.545512
1,1,similar,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,/tscc/lustre/ddn/scratch/kflanagan/plip_plop_r...,12955,300,ready,4.686861


Selected: PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ
Windows: 12,955; positions per window: 300
Selection uses window count, not compressed file size.


In [6]:
def load_comparison(path, selected):
    arrays, library_sizes = {}, {}
    with np.load(path, allow_pickle=False) as archive:
        n_windows = int(selected["n_windows"])
        for group in ("ip_A", "in_A", "ip_B", "in_B"):
            signal_key = group.replace("_", "_signals_", 1)
            shape = npz_array_shape(path, signal_key)
            if len(shape) != 3 or shape[0] != n_windows or shape[2] != 300 or shape[1] < 1:
                raise ValueError(f"{signal_key} must have shape (windows, replicates, 300); found {shape}")
            arrays[group] = np.asarray(archive[signal_key], dtype=np.float32)
            sizes = np.asarray(archive[f"{group}_library_sizes"], dtype=float)
            if sizes.shape != (shape[1],) or not np.all(np.isfinite(sizes) & (sizes > 0)):
                raise ValueError(f"Invalid library sizes for {group}: {sizes}")
            library_sizes[group] = sizes

        if arrays["in_A"].shape[1] != 1 or arrays["in_B"].shape[1] != 1:
            raise ValueError("The current enrichment workflow expects one IN replicate per experiment.")

        metadata = pd.DataFrame({key: archive[key] for key in ("chrom", "start", "end", "strand", "region_id", "block_number")})

    if len(metadata) != n_windows or not (metadata["end"] - metadata["start"]).eq(300).all():
        raise ValueError("Coordinate metadata must describe the same 300-nt windows as the signal arrays.")
    metadata["chrom"] = metadata["chrom"].astype(str)
    metadata["strand"] = metadata["strand"].astype(str)
    if not metadata["strand"].isin(["+", "-"]).all():
        raise ValueError("Unexpected strand in the input metadata.")
    metadata.insert(0, "window_idx", np.arange(n_windows))
    metadata.insert(0, "comparison_id", int(selected["comparison_id"]))
    metadata["comparison_key"] = comparison_key
    metadata["experiment_A"] = selected["experiment_A"]
    metadata["experiment_B"] = selected["experiment_B"]
    metadata["category"] = selected["category"]
    metadata["window_key"] = [f"{comparison_key}:w{index:07d}" for index in range(n_windows)]
    return arrays, library_sizes, metadata


started = perf_counter()
raw_arrays, library_sizes, window_metadata = load_comparison(INPUT_PATH, selected)
pseudocounts = {group: analysis_utils.calculate_rpm_pseudocount(sizes) for group, sizes in library_sizes.items()}
phase_timings["load_seconds"] = perf_counter() - started

print(f"Loaded {len(window_metadata):,} windows in {phase_timings['load_seconds']:.2f} s")
print(f"Raw arrays: {sum(array.nbytes for array in raw_arrays.values()) / 1e9:.3f} GB")
display(window_metadata.head())

Loaded 12,955 windows in 0.22 s
Raw arrays: 0.093 GB


,comparison_id,window_idx,chrom,start,end,strand,region_id,block_number,comparison_key,experiment_A,experiment_B,category,window_key
0,1,0,chr1,630973,631273,+,0,0,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,similar,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ:w...
1,1,1,chr1,631162,631462,+,0,1,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,similar,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ:w...
2,1,2,chr1,631648,631948,+,1,0,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,similar,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ:w...
3,1,3,chr1,631837,632137,+,1,1,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,similar,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ:w...
4,1,4,chr1,632226,632526,+,2,0,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ,PUM1_K562_ENCSR308YNT,PUM2_K562_ENCSR661ICQ,similar,PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ:w...


## Select the pilot

Each run has a separate output directory. Saved signals include only processed pilot windows; use the archive's `window_idx` array to map them to the source.

In [7]:
if PILOT_WINDOWS < 1 or BASELINE_CHECK_WINDOWS < 0:
    raise ValueError("Check PILOT_WINDOWS and BASELINE_CHECK_WINDOWS.")
pilot_indices = np.unique(np.linspace(0, len(window_metadata) - 1, min(PILOT_WINDOWS, len(window_metadata)), dtype=int))
actual_workers = pipeline_runner.worker_count(N_WORKERS, len(pilot_indices))
RUN_DIR = OUTPUT_ROOT / f"{comparison_key}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
pipeline_runner.write_tsv(window_metadata.loc[window_metadata["window_idx"].isin(pilot_indices)], RUN_DIR / "pilot_windows.tsv")
pipeline_runner.write_tsv(candidates, RUN_DIR / "comparison_candidates.tsv")
print(f"Pilot: {len(pilot_indices):,} / {len(window_metadata):,} windows, {actual_workers} worker(s)")
print(f"Results: {RUN_DIR}")

Pilot: 256 / 12,955 windows, 8 worker(s)
Results: /tscc/projects/ps-yeolab3/kflanagan/plip_plop/matrix_speed_up_test/optimized_pipeline/pilot_results/PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ_20261027_140128_879191


## Optional baseline comparison

Run the same small subset through both implementations on one core. All saved mode metrics and selection summaries must match exactly. These timings are separate from the pilot. Set `BASELINE_CHECK_WINDOWS=0` to skip.

The standalone `validate_equivalence.py` also compares full masks, ridges, island membership, and lag choices on synthetic cases.

In [8]:
baseline_report = None
if BASELINE_CHECK_WINDOWS:
    checks = pilot_indices[np.unique(np.linspace(0, len(pilot_indices) - 1, min(BASELINE_CHECK_WINDOWS, len(pilot_indices)), dtype=int))]
    reference_modes, reference_windows, _ = pipeline_runner.run_windows(
        raw_arrays, library_sizes, window_metadata, checks, configuration,
        RUN_DIR / "baseline_check" / "reference", n_workers=1, utils_path=REFERENCE_UTILS_PATH)
    optimized_modes, optimized_windows, _ = pipeline_runner.run_windows(
        raw_arrays, library_sizes, window_metadata, checks, configuration,
        RUN_DIR / "baseline_check" / "optimized", n_workers=1, utils_path=UTILS_PATH)
    pd.testing.assert_frame_equal(reference_modes, optimized_modes, check_exact=True)
    summary_columns = ["window_idx", "status", "n_modes", "n_fitted_modes", "n_islands", "n_provisional_modes",
                       "evaluated_lags", "central_gate", "central_gate_reason"]
    pd.testing.assert_frame_equal(reference_windows[summary_columns], optimized_windows[summary_columns], check_exact=True)
    before = float(reference_windows["total_seconds"].sum())
    after = float(optimized_windows["total_seconds"].sum())
    baseline_report = dict(n_windows=len(checks), exact_metrics_match=True,
        reference_compute_seconds=before, optimized_compute_seconds=after, single_core_speedup=before / after if after else None)
    print("Baseline comparison passed.")
    display(pd.DataFrame([baseline_report]))
    (RUN_DIR / "baseline_comparison.json").write_text(json.dumps(baseline_report, indent=2))

Windows (1 workers):   0%|          | 0/8 [00:00<?, ?window/s]

Windows (1 workers):   0%|          | 0/8 [00:00<?, ?window/s]

Baseline comparison passed.


,n_windows,exact_metrics_match,reference_compute_seconds,optimized_compute_seconds,single_core_speedup
0,8,True,1.947745,0.228683,8.517238


## Run the pilot

Only the selected indices are submitted. At most two tasks per worker are pending. Tables are sorted by original window index, so parallel completion cannot change mode identifiers or result order. The parent process writes checkpoints and error logs.

In [9]:
mode_metrics, window_timings, pilot_report = pipeline_runner.run_windows(
    raw_arrays, library_sizes, window_metadata, pilot_indices, configuration,
    RUN_DIR, n_workers=N_WORKERS, utils_path=UTILS_PATH
)
phase_timings["pilot_wall_seconds"] = pilot_report["wall_seconds"]

Windows (8 workers):   0%|          | 0/256 [00:00<?, ?window/s]

## Estimate runtime, then stop

The first projection scales total pilot wall time, including worker startup and checkpoint costs. The second pays the time to the first result once and extrapolates the subsequent completion rate. Both use this pilot's actual worker count.

These are estimates from the sampled window mix, not guaranteed runtimes. Summed worker compute time differs from elapsed wall time. Input loading is recorded separately and excluded from these analysis-time projections.

In [10]:
total = len(window_metadata)
processed = pilot_report["n_processed"]
wall = pilot_report["wall_seconds"]
first = pilot_report["first_result_seconds"] or 0.0
linear_estimate = wall * total / processed
steady_estimate = first + (total - 1) * (wall - first) / (processed - 1) if processed > 1 else linear_estimate

estimate = dict(total_windows=total, pilot_windows=processed, workers=pilot_report["n_workers"],
    pilot_wall_seconds=wall, pilot_modes=len(mode_metrics), errors=pilot_report["n_errors"],
    estimated_full_minutes=linear_estimate / 60,
    estimated_full_minutes_startup_adjusted=steady_estimate / 60,
    windows_per_wall_second=processed / wall)
display(pd.DataFrame([estimate]))
display(window_timings[["enrichment_seconds", "islands_seconds", "line_metrics_seconds", "total_seconds"]].agg(["sum", "mean", "median", "max"]).T)
display(window_timings.nlargest(10, "total_seconds")[["window_idx", "chrom", "start", "status", "n_modes",
    "n_islands", "evaluated_lags", "total_seconds"]])

manifest = dict(kind="pilot_only", pilot_finished=pilot_report["finished"], full_run_started=False,
    input_path=str(INPUT_PATH.resolve()), input_bytes=INPUT_PATH.stat().st_size, input_mtime_ns=INPUT_PATH.stat().st_mtime_ns,
    utils_sha256=hashlib.sha256(UTILS_PATH.read_bytes()).hexdigest(),
    reference_utils_sha256=hashlib.sha256(REFERENCE_UTILS_PATH.read_bytes()).hexdigest(),
    configuration=configuration, pilot_indices=pilot_indices.tolist(), phase_timings=phase_timings,
    estimates=estimate, baseline_comparison=baseline_report, python=platform.python_version(),
    updated_utc=datetime.now(timezone.utc).isoformat())
(RUN_DIR / "pilot_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"Pilot complete. No additional windows will run. Results: {RUN_DIR}")

,total_windows,pilot_windows,workers,pilot_wall_seconds,pilot_modes,errors,estimated_full_minutes,estimated_full_minutes_startup_adjusted,windows_per_wall_second
0,12955,256,8,3.197844,431,0,2.69714,1.447758,80.053925


,sum,mean,median,max
enrichment_seconds,0.094515,0.000369,0.000330,0.005063
islands_seconds,8.380343,0.032736,0.027437,0.121211
line_metrics_seconds,0.512147,0.002001,0.001398,0.010719
total_seconds,8.989869,0.035117,0.029843,0.123541


,window_idx,chrom,start,status,n_modes,n_islands,evaluated_lags,total_seconds
156,7924,chr21,43565319,ok,1,20,129,0.123541
205,10414,chr6,135181260,ok,9,15,202,0.108862
9,457,chr1,113124498,ok,1,10,109,0.107644
84,4267,chr15,44660327,ok,9,21,198,0.106668
13,660,chr1,201982746,ok,9,22,185,0.103315
16,812,chr1,1640258,ok,1,20,146,0.098265
24,1219,chr1,109064324,ok,1,15,155,0.092001
129,6553,chr19,5611274,ok,8,18,295,0.089513
132,6705,chr19,15239539,ok,2,15,258,0.089267
68,3454,chr12,121430106,ok,1,13,204,0.085359


Pilot complete. No additional windows will run. Results: /tscc/projects/ps-yeolab3/kflanagan/plip_plop/matrix_speed_up_test/optimized_pipeline/pilot_results/PUM1_K562_ENCSR308YNT__PUM2_K562_ENCSR661ICQ_20261027_140128_879191


The notebook stops here. It contains no automatic full-dataset, scatter, or clustering cell.

`mode_metrics.tsv`, `window_timings.tsv`, `enrichment_signals.npz`, and the manifests describe the pilot. Angle/displacement retain the prototype's bounding-box-relative definitions; `total_shape_adjusted_signal` remains the selector's score.

Use the duplicated island-identification and correspondence-line notebooks for visual spot checks. Their plotting code is retained.